In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load dataset
df = pd.read_csv(r"E:\Digital_Behavior_Fingerprint\data\raw\DSL-StrongPasswordData.csv")

# 2. Separate identifiers
id_columns = ["subject", "sessionIndex", "rep"]

# 3. Identify behavioural features
behaviour_features = [col for col in df.columns if col not in id_columns]

# 4. Group features
hold_features = [col for col in behaviour_features if col.startswith("H.")]
dd_features = [col for col in behaviour_features if col.startswith("DD.")]
ud_features = [col for col in behaviour_features if col.startswith("UD.")]

print("Hold features:", len(hold_features))
print("DD features:", len(dd_features))
print("UD features:", len(ud_features))

# 5. Calculate subject-level statistics
mean_features = df.groupby("subject")[behaviour_features].mean()
median_features = df.groupby("subject")[behaviour_features].median()
std_features = df.groupby("subject")[behaviour_features].std()
min_features = df.groupby("subject")[behaviour_features].min()
max_features = df.groupby("subject")[behaviour_features].max()

# 6. Create fingerprint features
fingerprint = pd.DataFrame(index=df["subject"].unique())

# Average timing behaviour
fingerprint["avg_hold_time"] = df.groupby("subject")[hold_features].mean().mean(axis=1)
fingerprint["avg_dd_time"] = df.groupby("subject")[dd_features].mean().mean(axis=1)
fingerprint["avg_ud_time"] = df.groupby("subject")[ud_features].mean().mean(axis=1)

# Timing variability
fingerprint["hold_time_std"] = df.groupby("subject")[hold_features].std().mean(axis=1)
fingerprint["dd_time_std"] = df.groupby("subject")[dd_features].std().mean(axis=1)
fingerprint["ud_time_std"] = df.groupby("subject")[ud_features].std().mean(axis=1)

# Minimum and maximum timing
fingerprint["min_hold_time"] = df.groupby("subject")[hold_features].min().mean(axis=1)
fingerprint["max_hold_time"] = df.groupby("subject")[hold_features].max().mean(axis=1)

# 7. Calculate approximate typing speed
fingerprint["typing_speed"] = 1 / (
    fingerprint["avg_hold_time"] + fingerprint["avg_dd_time"]
)

# 8. Add subject identifier
fingerprint.insert(0, "subject", fingerprint.index)

# 9. Reset index
fingerprint = fingerprint.reset_index(drop=True)

# 10. Display fingerprint
print("\nFingerprint shape:", fingerprint.shape)
display(fingerprint.head())

# 11. Check missing values
print("\nMissing values:")
print(fingerprint.isnull().sum())

# 12. Check statistics
print("\nFingerprint statistics:")
display(fingerprint.describe())

# 13. Visualize fingerprint features
fingerprint_features = fingerprint.drop(columns=["subject"])

fingerprint_features.hist(
    figsize=(15, 10),
    bins=20
)

plt.tight_layout()
plt.show()

# 14. Compare a few subjects
sample_subjects = fingerprint["subject"].head(5)

comparison = fingerprint[
    fingerprint["subject"].isin(sample_subjects)
]

display(comparison)

# 15. Save processed fingerprint dataset
fingerprint.to_csv(
    "../data/processed/keystroke_fingerprints.csv",
    index=False
)

print("\nFingerprint dataset saved successfully.")